# Logistic birth rate - Inference Quality Analyses

In this notebook we use synthetic viral read count data from a fully-parameterised toy population to theoretically assess
 - (1) the practical identifiability and 
 - (2) the quality of parameter inference

when the rodent population are assumed to follow the dynamics of the SIR algorithm with a logistic birth term rate. If the estimates of the population model parameters are close to the true model parameter values that characterise the toy population in the first place, this implies the validity of inferential approach, and therefore lend credibility to the results produced when the same pipeline is applied to metaviromic datasets collected through wildlife studies, as done in _James Hay et al. (2021)[1]_.

Similar to field studies, random samples of rodents are drawn from the simulated toy population at predifined sampling times, which satisfy the following:
 - same total number of rodents sampled at each time point;
 - the sampled individuals can be either susceptible (S), infected (I) or recovered (R), with no predefined quantities of each;
 - all individuals sampled are born and alive at the time of sampling.

For each of the sampled individuals, we use the SIR model's embedded `viral_read_model` to produce viral read count data, similar to what data is produced from the field studies (byproduct in our analyses, ground truth in real studies).

For the parameter inference we follow an optimisation approach, using the Bare-bones CMA-ES method from *Pints [2]* for parameter inference.

We replicate these analyses for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

**************
### References
[1] James A. Hay et al., _Estimating epidemiologic dynamics from cross-sectional viral load distributions_. Science373,**eabh0635(2021)**. DOI:10.1126/science.abh0635

[2] Clerx, M., Robinson, M., Lambert, B., Lei, C. L., Ghosh, S., Mirams, G. R., & Gavaghan, D. J.,
_Probabilistic Inference on Noisy Time Series (PINTS)_.
Journal of Open Research Software (2019), 7(1), 23. DOI:10.5334/jors.252

In [1]:
# Load necessary libraries
import numpy as np
import pandas as pd
from scipy.stats import multinomial, skew, gumbel_r
import math
import metavirommodel as mm
import metavirommodel.inference as mmi
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pints
from matplotlib import pyplot as plt
import pints.plot

# Choose array of colours for graphs and compartments names
colours = ['blue', 'red', 'green', 'purple', 'orange', 'black', 'gray', 'pink']
compartments = ['S', 'I', 'R']

# Set random seed
np.random.seed(270)

## Gillespie stochastic SIR algorithm with logistic birth term rate

#### Define rodent population

In [2]:
# Set initial reproduction number
R_0 = 3

# Set initial population state S - I - R
N_init = 200
# S_init = int(N_init / R_0)
S_init = 180
I_init = N_init - S_init
R_init = 0
initial_population = [S_init, I_init, R_init]

# Set birth rate
theta = 0.05

# Set death rates
mu = 0.002
nu = 0.003

# Set transition rates
infect_period = 30
beta =  R_0 / infect_period
gamma = 1 / infect_period

# Coalesce into paramater vector
parameters = initial_population
parameters.extend([theta, mu, nu, beta, gamma])

# Instantiate algorithm
algorithm = mm.LogisticGrowthMetaviromodel(carrying_capacity=400)

# Select start and end times
start_time = 1
end_time = 360

times = list(range(start_time, end_time+1))

# Select number of experiments
num_experiments = 1

output_algorithm = []

S_history_algorithm = []
I_history_algorithm = []
R_history_algorithm = []

I_times_history_algorithm = []
R_times_history_algorithm = []

for _ in range(num_experiments):
    output, S_history, I_history, R_history, I_times_history, R_times_history = algorithm.simulate_fixed_times(parameters, start_time, end_time)
    output_algorithm.append(output)

    S_history_algorithm.append(S_history)
    I_history_algorithm.append(I_history)
    R_history_algorithm.append(R_history)

    I_times_history_algorithm.append(I_times_history)
    R_times_history_algorithm.append(R_times_history)

output_algorithm = np.asarray(output_algorithm)

### Plot output of Gillespie for the different compartments

In [3]:
# Trace names - represent the type of individuals for the simulation
trace_name = ['{}'.format(s) for s in compartments]

# Names of panels
panels = ['{} only'.format(s) for s in compartments] + ['Total Population']

fig = go.Figure()
fig = make_subplots(rows=int(np.ceil(len(panels)/2)), cols=2, subplot_titles=tuple('{}'.format(p) for p in panels))

# Add traces to the separate counts panels
for s, spec in enumerate(compartments):
    fig.add_trace(
        go.Scatter(
            y=np.mean(output_algorithm[:, :, s], axis=0).tolist(),
            x=times,
            mode='lines',
            name=trace_name[s],
            line_color=colours[s]
        ),
        row= int(np.floor(s / 2)) + 1,
        col= s % 2 + 1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(np.sum(output_algorithm, axis=2), axis=0).tolist(),
        x=times,
        mode='lines',
        name='Total Population',
        line_color='black'
    ),
    row= 2,
    col= 2
)

# Add axis labels
fig.update_layout(
    title='Counts of compartments over time:<br>IC = {}, θ = {}, μ = {}, v = {}, β = {:.2f}, γ = {:.2f}'.format(parameters[0:3], parameters[3], parameters[4], parameters[5], parameters[6], parameters[7]),
    width=1100, 
    height=600,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis2=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis3=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis3=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis4=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis4=dict(
        linecolor='black',
        title = 'Individuals')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Logistic_VR_SIR-gillespie.pdf')
fig.show()

## Produce Viral read counts values

In [4]:
# Set parameter for the viral read counts model
t_eclipse = 3  # (0 days) Time from infection to initial viral growth
t_peak = 7  # (5 days ) Time from initial viral growth to peak viral load
t_switch = 5  # (9.38 days) Time from peak viral load to secondary waning phase
t_mod = 15  # (14 days) time from secondary waning phase until gumbel distribution reaches its min scale parameter
t_LOD = math.inf # ( inf days ) Time from infection until modal read counts value is equal to the limit of detection

sigma_obs = .25  # Initial scale parameter for the Gumbel distribution until a=teclipse+tpeak+tswitch
s_mod = 0.4  # 0.4 multiplicative factor applied to scale paramter for the Gumble distrbution - starting at t_eclipse + t_peak + t_switch + t_scle
v_zero = 2  # read counts value at time of infection
v_peak = 388  # (20) Modal read counts value at peak viral load
v_switch = 18  # (33) Modal read counts value at a = teclipse + tpeak + tswitch
v_LOD = 2  # Limit of detection of read counts value

parameters_vl = [
    t_eclipse, t_peak, t_switch, t_mod, t_LOD,
    v_zero, v_peak, v_switch, v_LOD,
    s_mod, sigma_obs]

# Set read counts value for the suceptible and recovered individuals
VR_susc = 2

### Plot Viral read Model

In [5]:
time_from_infec = np.arange(1, 50)
vr_val = []

for ti in time_from_infec:
    ti_vr_val = []
    for _ in range(10000):
        ti_vr_val.append(algorithm.viral_read_model(parameters_vl, ti))
    vr_val.append(ti_vr_val)

vr_val = np.asarray(vr_val)

In [6]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=time_from_infec,
        x=np.mean(vr_val, axis=1),
        mode='lines',
        name='Mean Viral read',
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        y=time_from_infec.tolist() + time_from_infec.tolist()[::-1],
        x=np.quantile(vr_val, 0.975, axis=1).tolist() + np.quantile(vr_val, 0.025, axis=1).tolist()[::-1],
        mode='lines',
        fill='toself',
        fillcolor='blue',
        line_color='blue',
        opacity=0.3,
        showlegend=False,
    )
)

# Add axis labels
fig.update_layout(
    width=500, 
    height=500,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Mean Viral read',
        autorange='reversed'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Time since infection'),
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Logistic-Viral_read_model.pdf')
fig.show()

### Compute the history of recovered individuals that fully clear the virus and generation times distribution

#### 0 = 'not cleared'; 1 = 'cleared'

In [7]:
# Daily probability of recovered fully clearing the virus
p_addl = 0.2

R_history_clear_algorithm = []

# Go through each run experiment
for _ in range(num_experiments):
    R_history_clear = []

    # Go through each recorded day
    for t, time in enumerate(times):
        current_clear_status = []

        # If there are any recovered individual
        if len(R_times_history_algorithm[_][t]) > 0:
            # Go through each of them and
            for ind, ind_ID in enumerate(R_history_algorithm[_][t]):
                clear_status = 0

                # If they have previously cleared the virus they signal that
                if ind_ID in R_history_algorithm[_][t-1] and R_history_clear[-1][R_history_algorithm[_][t-1].index(ind_ID)] == 1:
                    clear_status = 1
                # if not, they could do it today, if their time since infection exceeds teclipse + tpeak + tswitch
                elif time > R_times_history_algorithm[_][t][ind] + t_eclipse + t_peak + t_switch:
                    clear_status = 1 - np.random.binomial(1, p = (1-p_addl)**(
                        time - R_times_history_algorithm[_][t][ind] - t_eclipse - t_peak - t_switch))

                current_clear_status.append(clear_status)

        R_history_clear.append(current_clear_status)                

    R_history_clear_algorithm.append(R_history_clear)

In [8]:
# Compute the generation times distribution, which also follows a
# left-skewed Gumbel distribution
generation_times = []

for _ in range(70):
    if _ < t_eclipse + t_peak + t_switch:
        generation_times.append(
            1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            ))
        
    else:
        generation_times.append(
            (1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            )) * (1-p_addl)**(_ - t_eclipse - t_peak - t_switch))

#### Plot generation times

In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=time_from_infec,
        y=generation_times,
        mode='lines',
        name='Generation times',
        showlegend=False,
    )
)

fig.show()

## Parameter inference
In this section we test the quality of parameter inference for an optimisation approach, using the Bare-bones CMA-ES method from *Pints [2]* for single parameter inference, for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

#### Sample individuals with specific frequencies and in specific batch sizes

In [10]:
freq_samplying_range = [3, 5, 7, 15, 21, 30]
sample_size_range = [20, 30, 40, 50]

#### Method to create viral read data and ground truth

In [11]:
def sensitivity_analysis_run(sample_points, sample_size):
    vr_values = []
    vr_infec = []

    vr_susc_ids = []
    vr_infec_ids = []
    vr_recov_ids = []

    vr_time_of_recov_infec = []
    vr_time_of_infec = []
    vr_time_since_infec = []

    for _ in range(num_experiments):
        experiment_vr_values = []
        experiment_infec = []

        experiment_susc_ids = []
        experiment_infec_ids = []
        experiment_recov_ids = []

        experiment_time_of_recov_infec = []
        experiment_time_of_infec = []
        experiment_time_since_infec = []
        for time in sample_points:
            # Identify the current infections at the specified timepoint
            current_susceptibles = S_history_algorithm[_][time-1]
            current_infections = I_history_algorithm[_][time-1]
            current_recovered = R_history_algorithm[_][time-1]
            current_infection_times = I_times_history_algorithm[_][time-1]
            current_recov_infection_times = R_times_history_algorithm[_][time-1]
            current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]

            # Sample without replacement the sample_size individuals and
            # determine their time since infection to produce Ct values
            number_selected_susc, number_selected_infec, number_selected_rec = \
                multinomial.rvs(
                    n=sample_size,
                    p=output_algorithm[_, time-1, :]/np.sum(output_algorithm[_, time-1, :])) # determine how many of those sampled are S, I and R

            # First add the Ct values for the sampled susceptibele and recovered individuals
            sampled_vr_values = [VR_susc] * number_selected_susc

            selected_individuals_susc_ids = np.random.choice(
                    current_susceptibles,
                    size=number_selected_susc,
                    replace=False).tolist() # determine the ids of those sampled Ss
            
            if len(current_recov_infection_times) > 0:
                # If we have at least one selected recovered
                selected_individuals_indices = np.random.choice(
                    range(len(current_recov_infection_times)),
                    size=number_selected_rec,
                    replace=False).tolist() # determine the indices of those sampled Rs
            
                selected_individuals_rec_ids = [current_recovered[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Rs
                selected_individuals_recov_infec_times = [current_recov_infection_times[_] for _ in selected_individuals_indices]

                sample_time_since_infec = time - selected_individuals_recov_infec_times # determine how long since infection for selected Rs

                # Determine the clearence of infection of those sampled Rs
                selected_individuals_clear_virus_status = [current_recov_clear_virus_status[_] for _ in selected_individuals_indices]

                # Run viral read model to determine individual viral read counts for each sample
                for i, ti in enumerate(sample_time_since_infec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti) * selected_individuals_clear_virus_status[i])

            elif number_selected_rec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_rec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
                
                sample_time_since_infec = np.zeros(number_selected_rec)
                selected_individuals_rec_ids = [] 
            else:
                sample_time_since_infec = []
                selected_individuals_rec_ids = []

            if len(current_infection_times) > 0:
                # If we have at least one selected infection
                selected_individuals_indices = np.random.choice(
                    range(len(current_infection_times)),
                    size=number_selected_infec,
                    replace=False).tolist() # determine the indices of those sampled Is
                
                # Determine the ids of those sampled Is
                selected_individuals_infec_ids = [current_infections[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Is
                selected_individuals_infec_times = [current_infection_times[_] for _ in selected_individuals_indices]
            
                sample_time_since_infec = time - selected_individuals_infec_times # determine how long since infection for selected Is

                # Run Ct model to determine individual Ct counts for each sample
                for ti in sample_time_since_infec:
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti))
            
            elif number_selected_infec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_infec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
                
                selected_individuals_infec_times = np.zeros(number_selected_infec)
                sample_time_since_infec = np.zeros(number_selected_infec)
                selected_individuals_infec_ids = [] 
            else:
                selected_individuals_infec_times = []
                sample_time_since_infec = []
                selected_individuals_infec_ids = [] 

            experiment_vr_values.append(sampled_vr_values)
            experiment_infec.append(number_selected_infec)
            
            experiment_susc_ids.append(selected_individuals_susc_ids)
            experiment_infec_ids.append(selected_individuals_infec_ids)
            experiment_recov_ids.append(selected_individuals_rec_ids)

            experiment_time_of_recov_infec.append(selected_individuals_recov_infec_times)
            experiment_time_of_infec.append(selected_individuals_infec_times)
            experiment_time_since_infec.append(sample_time_since_infec)
        
        vr_values.append(experiment_vr_values)
        vr_infec.append(experiment_infec)

        vr_susc_ids.append(experiment_susc_ids)
        vr_infec_ids.append(experiment_infec_ids)
        vr_recov_ids.append(experiment_recov_ids)

        vr_time_of_recov_infec.append(experiment_time_of_recov_infec)
        vr_time_of_infec.append(experiment_time_of_infec)
        vr_time_since_infec.append(experiment_time_since_infec)

    vr_values = np.asarray(vr_values)
    vr_infec = np.asarray(vr_infec)

    vr_time_of_infec_data = []

    for _ in range(num_experiments):
        experiment_vr_time_of_infec_data = pd.DataFrame(columns=['ID', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_vr_time_of_infec_data = pd.concat(
                [
                    experiment_vr_time_of_infec_data,
                    pd.DataFrame({
                        'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                        'Value': [400] * len(vr_susc_ids[_][t]) + vr_time_of_recov_infec[_][t] + vr_time_of_infec[_][t]
                    })
                ])
            
        vr_time_of_infec_data.append(experiment_vr_time_of_infec_data)

    vr_values_data = []

    for _ in range(num_experiments):
        experiment_vr_values_data = pd.DataFrame(columns=['ID', 'TimeOfSample', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_vr_values_data = pd.concat(
                [
                    experiment_vr_values_data,
                    pd.DataFrame({
                        'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                        'TimeOfSample': [time] * sample_size,
                        'Value': vr_values[_, t, :].tolist()
                    })
                ])
            
        vr_values_data.append(experiment_vr_values_data)

    mvr_inference = mmi.LogisticGrowthMVRVirReadInfer(algorithm, generation_times=generation_times)

    # Read Vireal read counts and Ct values data
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)

    R0_found = mvr_inference.optimisation_problem_setup()[0]

    shody_recov_freq = []

    for t in range(vr_values[0].shape[0]):
        shody_recov_freq.append((np.where((vr_values[0][t, :] > 130) & (vr_values[0][t, :] < 150))[0]).shape[0] /sample_size)

    return vr_values_data, R0_found, shody_recov_freq, vr_infec[0, :] / sample_size

In [12]:
# Transform birth rate and death rates into function format for inference method
parameters[3] = lambda _: theta
parameters[4] = lambda _: mu
parameters[5] = lambda _: nu

#### Method to run inference with viral read data and plot inferred trajectories against ground truth

In [13]:
def routine_run(freq_samplying, sample_size):
    sample_points = np.arange(20, 200, freq_samplying)

    # For each choice of sample size and frequency infer parameters: 
    vr_values_data, R0_found, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_size)

    mvr_inference = mmi.LogisticGrowthMVRVirReadInfer(algorithm, generation_times=generation_times)
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)
    mvr_inference._create_posterior()

    parameters[6] = R0_found[0] / infect_period

    theta_found = []
    infec_found = []
    for _ in range(1000):
        output_found = algorithm.simulate_fixed_times(parameters, start_time, end_time)[0]

        theta_found.append(np.divide(output_found[:, 1], np.sum(output_found, axis=1)))
        infec_found.append(output_found[:, 1])

    theta_found = np.array(theta_found)
    infec_found = np.array(infec_found)

    theta_found_mean = np.mean(theta_found, axis=0)
    theta_found_upper = np.quantile(theta_found, 0.975, axis=0)
    theta_found_lower = np.quantile(theta_found, 0.025, axis=0)

    infec_found_mean = np.mean(infec_found, axis=0)
    infec_found_upper = np.quantile(infec_found, 0.975, axis=0)
    infec_found_lower = np.quantile(infec_found, 0.025, axis=0)

    output_found_det = mvr_inference.loglikelihood._run_sir_model(
        parameters, np.arange(max(mvr_inference.loglikelihood._vr_sampled_times))
    )
    theta_found_det = np.divide(output_found_det[:, 1], np.sum(output_found_det, axis=1))
    infec_found_det = output_found_det[:, 1]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_freq_sample.tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq',
            line_color='blue'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=(1 - np.array(shody_recov_freq)).tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq (extreme I)',
            line_color='green'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=theta_found_upper.tolist() + theta_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'),
        )

    fig.write_image('images/Logistic_Viral_read_CredInt_Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1]).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=infec_found_upper.tolist() + infec_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'))

    fig.write_image('images/Total_Infec_Logistic_Viral_read_Sensitivity_Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

#### Run optimisation-based inference method for multiple sampling protcols

In [14]:
routine_run(freq_samplying_range[2], sample_size_range[0])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:158: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -1373.883 -1373.883   0:10.3
1     8     -1347.582 -1347.582   0:24.2
2     12    -1347.582 -1364.159   0:33.7
3     16    -1312.724 -1312.724   0:47.6
20    84    -1247.036 -1247.036   4:59.6
40    164   -1247.035 -1247.035  12:15.0
60    244   -1247.035 -1247.035  19:55.3
80    324   -1247.035 -1247.035  27:48.1
100   404   -1247.035 -1247.035  35:32.8
114   456   -1247.035 -1247.035  40:33.3
Halting: No significant change in best function evaluation for 100 iterations.
[2.22482378] -1247.0346850909248
Optimisation phase is finished.


In [15]:
routine_run(freq_samplying_range[2], sample_size_range[3])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:142: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:158: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -2933.712 -2933.712   0:53.8
1     8     -2933.712 -2948.08    1:48.3
2     12    -2924.105 -2924.105   2:41.4
3     16    -2924.105 -2930.371   3:34.7
20    84    -2924.037 -2924.037  18:38.8
40    164   -2924.037 -2924.037  35:48.7
60    244   -2924.037 -2924.037  52:49.3
80    324   -2924.037 -2924.037  69:39.5
100   404   -2924.037 -2924.037  86:45.7
103   412   -2924.037 -2924.037  88:26.5
Halting: No significant change in best function evaluation for 100 iterations.
[2.26951476] -2924.037062900017
Optimisation phase is finished.


In [16]:
routine_run(freq_samplying_range[5], sample_size_range[0])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:142: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:158: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -272.806  -272.806    0:05.1
1     8     -272.806  -276.9175   0:10.6
2     12    -272.806  -273.7327   0:14.2
3     16    -272.806  -272.8106   0:19.0
20    84    -272.7948 -272.7948   1:36.6
40    164   -272.7948 -272.7948   3:06.7
60    244   -272.7948 -272.7948   4:37.5
80    324   -272.7948 -272.7948   6:08.0
100   404   -272.7948 -272.7948   7:39.1
101   404   -272.7948 -272.7948   7:39.1
Halting: No significant change in best function evaluation for 100 iterations.
[2.29825428] -272.7947951560857
Optimisation phase is finished.


In [17]:
routine_run(freq_samplying_range[5], sample_size_range[3])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:142: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:158: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -622.8126 -622.8126   0:11.5
1     8     -622.8126 -625.9622   0:22.5
2     12    -622.8126 -625.9996   0:33.9
3     16    -621.6298 -621.6298   0:44.9
20    84    -621.6255 -621.6255   3:53.7
40    164   -621.6254 -621.6254   7:33.8
60    244   -621.6254 -621.6254  10:59.8
80    324   -621.6254 -621.6254  14:23.0
100   404   -621.6254 -621.6254  17:46.7
104   416   -621.6254 -621.6254  18:17.0
Halting: No significant change in best function evaluation for 100 iterations.
[2.3748381] -621.625445904575
Optimisation phase is finished.


## Plot viral read counts sampled values at each sampling point
We colour all sampled values at each time point according to the skewness of the sample

In [18]:
freq_samplying = 10
sample_size = 10

sample_points = np.arange(5, 200, freq_samplying)

# For each choice of sample size and frequency infer parameters: 
vr_values_data, R0_found, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_size)


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:142: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/328490559.py:158: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -452.6985 -452.6985   0:08.1
1     8     -452.6985 -453.2305   0:15.6
2     12    -450.0009 -450.0009   0:23.1
3     16    -450.0009 -450.0706   0:31.0
20    84    -449.9842 -449.9843   2:37.1
40    164   -449.9842 -449.9842   5:04.2
60    244   -449.9842 -449.9842   7:32.0
80    324   -449.9842 -449.9842   9:59.2
100   404   -449.9842 -449.9842  12:26.0
103   412   -449.9842 -449.9842  12:40.6
Halting: No significant change in best function evaluation for 100 iterations.
[2.28866424] -449.9842255418555
Optimisation phase is finished.


In [19]:
fig = go.Figure()
fig = make_subplots(rows=2, cols=1, subplot_titles=('Average VR value', 'Average Prevalence of Infections'))

vr_values = []
for t in vr_values_data[0]['TimeOfSample'].unique():
    vr_values.append(vr_values_data[0][vr_values_data[0]['TimeOfSample'] == t]['Value'].tolist())
vr_values = np.log(np.asarray(vr_values))
skews = skew(vr_values, axis=1)
print(skews)

for _ in range(sample_size):
    fig.add_trace(
        go.Scatter(
            y=vr_values[:, _].tolist(),
            x=sample_points,
            mode='markers',
            name='Average VR value',
            marker_line=dict(width=1.25, color='black'),
            marker_color=skews,
            marker_colorscale='BlueRed_r',
            marker_colorbar_title=dict(
                text='Skew',
                side='top'),
            marker_size=12,
            marker_opacity=0.6,
            marker_showscale=True,
            showlegend=False,
        ),
        row=1,
        col=1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(vr_values, axis=1).tolist(),
        x=sample_points,
        mode='markers',
        name='Median VR value',
        marker_color='black',
        marker_line=dict(width=4, color='black'),
        marker_symbol='line-ew',
        marker_size=16,
        showlegend=False,
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        y=infec_freq_sample.tolist(),
        x=sample_points,
        mode='lines+markers',
        connectgaps=True,
        name='Average Prevalence of Infections',
        marker_line=dict(width=1.25, color='black'),
        marker_color=skews,
        marker_colorscale='BlueRed_r',
        marker_colorbar_title=dict(
            text='Skew',
            side='top'),
        marker_opacity=0.6,
        marker_size=12,
        marker_showscale=True,
        line_color='black',
        showlegend=False,
    ),
    row=2,
    col=1
)

# Add axis labels
fig.update_layout(
    width=1100, 
    height=800,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        tickvals=sample_points.tolist(),
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Viral read value (e+03)'),
    xaxis2=dict(
        linecolor='black',
        tickvals=sample_points.tolist(),
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        tickformat = '.0%',
        title = 'Prevalence of Infections<br>in sample')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Logistic_Viral_read_values.pdf')
fig.show()

[ 2.66666667         nan  2.66666667  0.26660151  0.04806757  0.23762963
  0.571424    0.96522812 -0.82542168 -1.49166465  0.48901085  0.46800923
  2.51009487 -2.49235143 -2.4996979  -0.39611629 -1.46311563 -0.83919507
 -0.85565897 -1.47162418]


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_86116/1761918414.py:7: RuntimeWarning:

divide by zero encountered in log

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/scipy/stats/_stats_py.py:1127: RuntimeWarning:

invalid value encountered in subtract



#### Plot the proportion of infected in the sampled against the ground truth of the total population

In [20]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
        x=times,
        mode='lines',
        name='True freq',
        line_color='red'
    )
)

fig.add_trace(
    go.Scatter(
        y=infec_freq_sample.tolist(),
        x=sample_points,
        mode='lines',
        name='Sample freq',
        line_color='blue'
    )
)

# Add axis labels
fig.update_layout(
    title='Sensitivity analysis: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
    width=700, 
    height=400,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    )

fig.write_image('images/True_vs_Sampled_Freq_Infec_Logistic_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
fig.show()